# Neural Flows

The diea is to train a neural flow model that transforms the distribution of our morphological parameters $p(x)$ into a Gaussian distribution $p(z)$. Then the latent space of that model should learn something about the dataset.

We do this conditionally on a few parameters:
* Brightness: r-magnitude
* Nuisance parameters: SNR, PSF FWHM
This way the question is, given this r-band magnitude and these nuisance parameters, what is the probability that a galaxy should have this set of morphological measurements?

Irregular galaxies will then be flagged as artifacts (e.g., different ellipticity metrics don't match; asymmetry is high; etc.)

---

#### Imports

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import h5py
import jax
import jax.numpy as jnp
import equinox as eqx
import optax
from flowjax.flows import coupling_flow
from flowjax.bijections import RationalQuadraticSpline
from flowjax.distributions import Normal
from flowjax.train import fit_to_data
from pathlib import Path
import time
%matplotlib inline

### Load the data

The data is stored as an HDF5 file with the following datasets:
* `meta`: tile names, galaxy ID from the source catalog, ra, dec, PSF FWHM
* `condition`: variables we condition on - r-band magnitude, FWHM, SN, sky median and sky RMS
* `training`: variables we train on - morphology, radius, cutout sizes.
* `masks`: masks of bad variable ranges. Also something we train on (this will be a binary flag).

dataset = h5py.File('../catalogs/dataset.h5', 'r')

## Setting up the pipeline

### Model architecture

First, let's configure our neural spline flow. We want a small-ish network that can fit on an 8GB GPU.


In [2]:
DATA_PATH = "../catalogs/dataset.h5"
CHECKPOINT_DIR = Path("../data/flow_checkpoints")
CHECKPOINT_DIR.mkdir(exist_ok=True)

# Architecture
N_FLOW_LAYERS = 10
N_NN_WIDTH = 128
N_NN_DEPTH = 3
N_SPLINE_BINS = 16
SPLINE_RANGE = 10.0  
# should match the preprocessed data range at least approximately, with some small exceptions we can extrapolate to

# Training
BATCH_SIZE = 2048   # reduce if OOM
LEARNING_RATE = 5e-4
N_EPOCHS = 20
SEED = 0

# Data subset for quick iteration; set to None to use all
N_SUBSET = 5_000_000

### Loading in the data

This is final pre-processing of the data.

1. Load the entire HDF5 file into memory (25GB, we can afford).
    * Optional: select a subset to train on
2. Stack the training set columns and the masks together into a single training data array

In [4]:
def load_data(path, n_subset=None):
    """Load all features, masks, and conditioning into memory. 
    n_subset: how many galaxies to try training on. If None, train on all.
    """
    print(f"Loading {path}...")
    with h5py.File(path, "r") as f:
        feature_keys = sorted(f["training"].keys())
        mask_keys = sorted(f["masks"].keys())
        cond_keys = sorted(f["condition"].keys())

        N = f[f"training/{feature_keys[0]}"].shape[0]
        if n_subset is not None and n_subset < N:
            idx = np.sort(np.random.default_rng(SEED).choice(N, n_subset, replace=False))
            print(f"  Subsetting to {n_subset:,} of {N:,} rows")
        else:
            idx = slice(None)
            print(f"  Loading all {N:,} rows")

        # Stack columns into arrays. h5py fancy indexing is faster with sorted indices.
        features = np.stack([f[f"training/{k}"][idx] for k in feature_keys], axis=1)
        masks = np.stack([f[f"masks/{k}"][idx] for k in mask_keys], axis=1)
        condition = np.stack([f[f"condition/{k}"][idx] for k in cond_keys], axis=1)

    print(f"  features: {features.shape}, masks: {masks.shape}, condition: {condition.shape}")
    # Combine features and masks into single feature matrix
    x = np.concatenate([features, masks.astype(np.float32)], axis=1).astype(np.float32)
    c = condition.astype(np.float32)

    return x, c, feature_keys, mask_keys, cond_keys


In [ ]:
X, Y, feature_keys, mask_keys, cond_keys = load_data(DATA_PATH, n_subset=5_000_000)

Loading ../catalogs/dataset.h5...
  Subsetting to 5,000,000 of 44,481,191 rows


In [ ]:

# ---------------------------------------------------------------
# Train/val split + standardize conditioning
# ---------------------------------------------------------------

def prepare_arrays(x, c, val_frac=0.05, seed=SEED):
    rng = np.random.default_rng(seed)
    N = x.shape[0]
    perm = rng.permutation(N)
    n_val = int(val_frac * N)
    val_idx = perm[:n_val]
    train_idx = perm[n_val:]

    # Standardize conditioning using training stats (your features are
    # already on (-10, 10), but conditioning probably isn't)
    c_med = np.median(c[train_idx], axis=0)
    c_iqr = np.subtract(*np.percentile(c[train_idx], [75, 25], axis=0))
    c_iqr = np.where(c_iqr > 0, c_iqr, 1.0)  # safety
    c_std = (c - c_med) / c_iqr

    x_train, c_train = x[train_idx], c_std[train_idx]
    x_val, c_val = x[val_idx], c_std[val_idx]
    print(f"  Train: {x_train.shape[0]:,}, Val: {x_val.shape[0]:,}")
    return x_train, c_train, x_val, c_val, (c_med, c_iqr)

# ---------------------------------------------------------------
# Dequantization for binary mask columns
# ---------------------------------------------------------------

def dequantize_masks(x, n_continuous, key):
    """Add U[0,1) noise to binary mask columns to make them continuous."""
    n_total = x.shape[-1]
    n_masks = n_total - n_continuous
    noise = jax.random.uniform(key, shape=(x.shape[0], n_masks))
    masks_dequant = x[:, n_continuous:] + noise
    return jnp.concatenate([x[:, :n_continuous], masks_dequant], axis=-1)

# ---------------------------------------------------------------
# Build the flow
# ---------------------------------------------------------------

def build_flow(key, dim, cond_dim):
    """Conditional coupling flow with rational-quadratic splines."""
    return coupling_flow(
        key=key,
        base_dist=Normal(jnp.zeros(dim)),
        cond_dim=cond_dim,
        flow_layers=N_FLOW_LAYERS,
        nn_width=N_NN_WIDTH,
        nn_depth=N_NN_DEPTH,
        transformer=RationalQuadraticSpline(
            knots=N_SPLINE_BINS,
            interval=SPLINE_RANGE,
        ),
    )

# ---------------------------------------------------------------
# Training loop
# ---------------------------------------------------------------

def train(x_train, c_train, x_val, c_val, n_continuous, key):
    dim = x_train.shape[1]
    cond_dim = c_train.shape[1]

    key, subkey = jax.random.split(key)
    flow = build_flow(subkey, dim, cond_dim)

    optimizer = optax.adam(LEARNING_RATE)

    # JIT-compiled loss and step
    @eqx.filter_jit
    def loss_fn(flow, x, c):
        return -flow.log_prob(x, condition=c).mean()

    @eqx.filter_jit
    def step(flow, opt_state, x, c):
        loss, grads = eqx.filter_value_and_grad(loss_fn)(flow, x, c)
        updates, opt_state = optimizer.update(grads, opt_state, flow)
        flow = eqx.apply_updates(flow, updates)
        return flow, opt_state, loss

    opt_state = optimizer.init(eqx.filter(flow, eqx.is_inexact_array))

    N = x_train.shape[0]
    n_batches = N // BATCH_SIZE
    print(f"\n{n_batches} batches/epoch, batch size {BATCH_SIZE}\n")

    best_val_loss = float("inf")
    history = []

    for epoch in range(N_EPOCHS):
        t0 = time.time()
        key, perm_key = jax.random.split(key)
        perm = np.array(jax.random.permutation(perm_key, N))

        train_loss_sum = 0.0
        for b in range(n_batches):
            idx = perm[b * BATCH_SIZE : (b + 1) * BATCH_SIZE]
            x_batch = jnp.asarray(x_train[idx])
            c_batch = jnp.asarray(c_train[idx])

            # Dequantize mask columns each batch
            key, dq_key = jax.random.split(key)
            x_batch = dequantize_masks(x_batch, n_continuous, dq_key)

            flow, opt_state, loss = step(flow, opt_state, x_batch, c_batch)
            train_loss_sum += float(loss)

        train_loss = train_loss_sum / n_batches

        # Validation in chunks to avoid OOM
        val_losses = []
        for b in range(0, x_val.shape[0], BATCH_SIZE):
            x_b = jnp.asarray(x_val[b : b + BATCH_SIZE])
            c_b = jnp.asarray(c_val[b : b + BATCH_SIZE])
            key, dq_key = jax.random.split(key)
            x_b = dequantize_masks(x_b, n_continuous, dq_key)
            val_losses.append(float(loss_fn(flow, x_b, c_b)))
        val_loss = np.mean(val_losses)

        dt = time.time() - t0
        history.append({"epoch": epoch, "train": train_loss, "val": val_loss, "time": dt})
        print(f"epoch {epoch:3d}  train {train_loss:8.3f}  val {val_loss:8.3f}  ({dt:.0f}s)")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            eqx.tree_serialise_leaves(CHECKPOINT_DIR / "flow_best.eqx", flow)

    eqx.tree_serialise_leaves(CHECKPOINT_DIR / "flow_final.eqx", flow)
    return flow, history

# ---------------------------------------------------------------
# Run
# ---------------------------------------------------------------

if __name__ == "__main__":
    x, c, fkeys, mkeys, ckeys = load_data(DATA_PATH, n_subset=N_SUBSET)
    n_continuous = len(fkeys)  # 51 continuous features, 48 are masks
    print(f"  {n_continuous} continuous features, {len(mkeys)} mask features")

    x_train, c_train, x_val, c_val, cond_stats = prepare_arrays(x, c)

    # Save conditioning standardization for later inference
    np.savez(CHECKPOINT_DIR / "cond_stats.npz", med=cond_stats[0], iqr=cond_stats[1])

    key = jax.random.PRNGKey(SEED)
    flow, history = train(x_train, c_train, x_val, c_val, n_continuous, key)

    print(f"\nDone. Best val loss: {min(h['val'] for h in history):.3f}")